# Lesson 1.3.1 - Version Control, Git, Branching & PRs

## Learning Objectives
- Explain why Git workflow quality directly impacts AI system reliability.
- Use branch naming and PR templates to reduce delivery risk.
- Apply lightweight risk scoring for PR review prioritization.


## Why This Matters for AI Engineering
A model pipeline can fail in production because of one unsafe merge: schema changes, secret leaks, or unreviewed experiment code promoted to `main`.

Treat Git as an engineering control system, not a backup tool.


In [1]:
from dataclasses import dataclass
from typing import List


@dataclass
class FileChange:
    path: str
    lines_changed: int


def risk_score(changes: List[FileChange]) -> int:
    """Very small heuristic to prioritize PR review depth."""
    score = 0
    for ch in changes:
        if ch.path.startswith('infra/') or ch.path.startswith('serve/'):
            score += 4
        elif ch.path.startswith('data/'):
            score += 3
        elif ch.path.startswith('notebooks/'):
            score += 2
        else:
            score += 1

        if ch.lines_changed > 300:
            score += 2
    return score


changes = [
    FileChange('serve/predict_api.py', 120),
    FileChange('data/schema.yaml', 32),
    FileChange('notebooks/experiment.ipynb', 480),
]

score = risk_score(changes)
review_tier = 'high' if score >= 9 else 'medium' if score >= 5 else 'low'
print({'risk_score': score, 'review_tier': review_tier})


{'risk_score': 11, 'review_tier': 'high'}


## Reusable Branching Strategy Template
Use this starter when planning work:

```text
Branch name: feat/<feature-name> | fix/<bug-name>
Owner: <name>
Scope: <what changes and what does not>
Risk areas: <data, model, serving, infra>
Rollback plan: <git revert strategy + fallback artifact>
```

For AI repos, keep experiment branches short-lived and merge only reusable components.


In [2]:
def build_pr_template(problem: str, changes: list[str], metrics: dict, risks: list[str]) -> str:
    lines = [
        '# PR Summary',
        f'## Problem\n{problem}',
        '## Changes',
        *[f'- {c}' for c in changes],
        '## Validation Metrics',
        *[f'- {k}: {v}' for k, v in metrics.items()],
        '## Risks and Rollback',
        *[f'- {r}' for r in risks],
    ]
    return '\n'.join(lines)


pr_text = build_pr_template(
    problem='Prediction API fails when optional features are missing.',
    changes=['Added schema validation', 'Added default value policy', 'Added integration test'],
    metrics={'API failure rate': '2.3% -> 0.2%', 'p95 latency': '88ms -> 92ms'},
    risks=['Minor latency increase', 'Backward compatibility monitored for one week'],
)
print(pr_text)


# PR Summary
## Problem
Prediction API fails when optional features are missing.
## Changes
- Added schema validation
- Added default value policy
- Added integration test
## Validation Metrics
- API failure rate: 2.3% -> 0.2%
- p95 latency: 88ms -> 92ms
## Risks and Rollback
- Minor latency increase
- Backward compatibility monitored for one week


## Pull Request Checklist (Reusable)
- [ ] Problem statement is explicit and testable.
- [ ] Data/model/API contracts changed are documented.
- [ ] Repro steps include commands and seed/config.
- [ ] Security check done (no credentials, no sensitive data).
- [ ] Rollback path documented.
- [ ] Reviewer assigned based on risk tier.


In [3]:
checklist = {
    'problem_statement': True,
    'contract_notes': True,
    'repro_steps': True,
    'security_check': True,
    'rollback_plan': False,
}

ready = all(checklist.values())
print('Merge-ready:', ready)
if not ready:
    print('Missing:', [k for k, v in checklist.items() if not v])


Merge-ready: False
Missing: ['rollback_plan']


## Business Case Studies & Exceptions
### Case 1: Secret Leak in Notebook Output
A team accidentally pushed an API key in notebook output. The correct response is immediate key rotation, secret scanning, and PR policy tightening.

### Case 2: Model Contract Break from Unreviewed Merge
A model output schema changed (`score` -> `risk_score`) without integration tests. Downstream consumers crashed. Solution: contract tests + required reviewer for serving paths.

### Exception
For a solo prototype branch that is not user-facing, you can use lighter review rules, but still keep commit hygiene and a rollback checkpoint.


## Interview Questions & Answers
1. **Q:** Why is Git workflow critical in ML systems?  
   **A:** It provides traceability and rollback for code that affects data, features, and model behavior.
2. **Q:** What is a good branch naming pattern?  
   **A:** `feat/<intent>`, `fix/<issue>`, `chore/<maintenance>`.
3. **Q:** What makes a PR high risk?  
   **A:** Changes in serving, schema, auth, or large cross-cutting diffs.
4. **Q:** Why include metrics in PR descriptions?  
   **A:** To justify impact and avoid subjective merge decisions.
5. **Q:** What should rollback include?  
   **A:** Revert strategy and last known-good artifact/version.
6. **Q:** Trunk-based vs feature branching?  
   **A:** Trunk-based favors frequent integration; feature branching isolates work.
7. **Q:** Why avoid giant PRs?  
   **A:** They reduce review quality and increase regression risk.
8. **Q:** What is branch protection?  
   **A:** Rules requiring checks/review before merging into protected branches.
9. **Q:** How do you review notebook-heavy PRs?  
   **A:** Prefer extracting production logic into modules and reviewing notebook outputs minimally.
10. **Q:** What is one anti-pattern in AI Git workflows?  
   **A:** Merging experiments directly into production without contract and regression checks.
